In [0]:
sliver="abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/"
gold="abfss://gold@yuvastorage1adlsgen2.dfs.core.windows.net/"
input_path=f"{sliver}SalesLT"

In [0]:
display(dbutils.fs.ls(input_path))

path,name,size,modificationTime
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/Address/,Address/,0,1791700152000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/Customer/,Customer/,0,1791700157000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/CustomerAddress/,CustomerAddress/,0,1791700160000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/Product/,Product/,0,1791700164000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/ProductCategory/,ProductCategory/,0,1791700167000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/ProductDescription/,ProductDescription/,0,1791700170000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/ProductModel/,ProductModel/,0,1791700173000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/ProductModelProductDescription/,ProductModelProductDescription/,0,1791700176000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/SalesOrderDetail/,SalesOrderDetail/,0,1791700179000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/SalesOrderHeader/,SalesOrderHeader/,0,1791700182000


In [0]:
display(dbutils.fs.ls(f"{input_path}/Address"))

path,name,size,modificationTime
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/Address/_delta_log/,_delta_log/,0,1791700152000
abfss://sliver@yuvastorage1adlsgen2.dfs.core.windows.net/SalesLT/Address/part-00000-da107d9e-f761-488d-b904-1f28ef9b9b5b.c000.snappy.parquet,part-00000-da107d9e-f761-488d-b904-1f28ef9b9b5b.c000.snappy.parquet,35345,1791700153000


In [0]:
from pyspark.sql.functions import col

In [0]:
import re
for file_path in dbutils.fs.ls(input_path):
    df = spark.read.format('delta').load(f"{input_path}/{file_path.name.strip('/')}")
    for column_name in df.columns:
        new_column_name=re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', column_name)
        new_column_name=re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', new_column_name).lower()
        df=df.withColumnRenamed(column_name, new_column_name)

    table_name = file_path.name.strip("/")
    path=f"{gold}/SalesLT/{file_path.name}{table_name}"
    df.write.format('delta').mode('overwrite').save(path)